# 🚀 Boosting Models — PII Data Detection

Notebook ini menjalankan training & evaluasi **XGBoost** dan **LightGBM** untuk token-level PII detection.

**Data yang digunakan:**
- `data/processed/imbalance/` → train.json, val.json, test.json (distribusi asli)
- `data/processed/balance/` → train.json, val.json, test.json (sudah di-balance)

**4 varian model:**
1. LightGBM Imbalance (data dari `imbalance/`)
2. XGBoost Imbalance (data dari `imbalance/`)
3. LightGBM Balance (data dari `balance/`)
4. XGBoost Balance (data dari `balance/`)

**Output:**
- 4 CSV predictions di `results/predictions/`
- 4 JSON metrics di `results/metrics/`

> ⚠️ **Pastikan GPU accelerator ENABLED di Kaggle settings (untuk XGBoost CUDA)**

## Cell 1 — Clone Repository & Install Dependencies

In [ ]:
import shutil, os
shutil.rmtree("/kaggle/working/PII-Data-Detection", ignore_errors=True)
!git clone -b https://github.com/yukakeren/PII-Data-Detection.git
%cd PII-Data-Detection

In [ ]:
!grep -v "pycrfsuite" requirements.txt > requirements_fixed.txt
!pip install -r requirements_fixed.txt -q
!pip install -r models/boosting/requirements.txt -q

## Cell 2 — Copy Dataset ke data/processed/

Data sudah di-split menjadi 2 folder:
- `data/processed/balance/` → data yang sudah di-balance
- `data/processed/imbalance/` → data distribusi asli

Masing-masing folder berisi `train.json`, `val.json`, dan `test.json`.

In [ ]:
import os

# Buat direktori yang diperlukan
for d in ["data/processed/balance", "data/processed/imbalance", "results/predictions", "results/metrics"]:
    os.makedirs(d, exist_ok=True)

# Copy data balance
!cp /kaggle/input/datasets/ericatriana/balance-data/train.json data/processed/balance/
!cp /kaggle/input/datasets/ericatriana/balance-data/val.json data/processed/balance/
!cp /kaggle/input/datasets/ericatriana/balance-data/test.json data/processed/balance/

# Copy data imbalance
!cp /kaggle/input/datasets/ericatriana/imbalance-data/train.json data/processed/imbalance/
!cp /kaggle/input/datasets/ericatriana/imbalance-data/val.json data/processed/imbalance/
!cp /kaggle/input/datasets/ericatriana/imbalance-data/test.json data/processed/imbalance/

print("\n📁 Balance data:")
!ls -la data/processed/balance/
print("\n📁 Imbalance data:")
!ls -la data/processed/imbalance/

## Cell 3 — Import Modules & Setup

In [ ]:
import os, sys, warnings
import numpy as np
warnings.filterwarnings("ignore")

# Setup path
sys.path.insert(0, "/kaggle/working/PII-Data-Detection")
os.chdir("/kaggle/working/PII-Data-Detection")

# Import project modules
from src.data_loader import DataLoader
from models.boosting.feature_extraction import (
    build_feature_matrix, encode_labels,
    compute_class_weights, get_sample_weights, quick_f1_pii,
)
from models.boosting.train_lightgbm import (
    train_lightgbm_imbalance, train_lightgbm_balance,
)
from models.boosting.train_xgboost import (
    train_xgboost_imbalance, train_xgboost_balance, create_label_remap,
)
from models.boosting.predict import run_prediction_pipeline

print("✅ Semua module berhasil di-import!")
print(f"Working dir: {os.getcwd()}")

## Cell 4 — Load Data & Feature Extraction

Load 6 file data dari 2 folder terpisah, lalu extract features:
- **Imbalance**: `data/processed/imbalance/` (train, val, test)
- **Balance**: `data/processed/balance/` (train, val, test)

> ⏱️ Proses ini memakan waktu ~10 menit (building features untuk 6 dataset)

In [ ]:
# ── Load data ──
print("📂 Loading data...")
loader = DataLoader()

# Imbalance data
train_imb = loader.load_raw_json("data/processed/imbalance/train.json")
val_imb   = loader.load_raw_json("data/processed/imbalance/val.json")
test_imb  = loader.load_raw_json("data/processed/imbalance/test.json")
print(f"  Imbalance — Train: {len(train_imb)} | Val: {len(val_imb)} | Test: {len(test_imb)} docs")

# Balance data
train_bal = loader.load_raw_json("data/processed/balance/train.json")
val_bal   = loader.load_raw_json("data/processed/balance/val.json")
test_bal  = loader.load_raw_json("data/processed/balance/test.json")
print(f"  Balance   — Train: {len(train_bal)} | Val: {len(val_bal)} | Test: {len(test_bal)} docs")

In [ ]:
# ── Build features ──
print("\n🔧 Building features untuk Imbalance data...")
X_train_imb, y_train_imb_str, _, vectorizer = build_feature_matrix(train_imb, fit_vectorizer=True)
X_val_imb,   y_val_imb_str,   _, _          = build_feature_matrix(val_imb,   vectorizer=vectorizer)
X_test_imb,  y_test_imb_str,  meta_test_imb, _ = build_feature_matrix(test_imb,  vectorizer=vectorizer)
print(f"  Imbalance shapes: Train={X_train_imb.shape}, Val={X_val_imb.shape}, Test={X_test_imb.shape}")

print("\n🔧 Building features untuk Balance data...")
X_train_bal, y_train_bal_str, _, _ = build_feature_matrix(train_bal, vectorizer=vectorizer)
X_val_bal,   y_val_bal_str,   _, _ = build_feature_matrix(val_bal,   vectorizer=vectorizer)
X_test_bal,  y_test_bal_str,  meta_test_bal, _ = build_feature_matrix(test_bal, vectorizer=vectorizer)
print(f"  Balance shapes:   Train={X_train_bal.shape}, Val={X_val_bal.shape}, Test={X_test_bal.shape}")

In [ ]:
# ── Encode labels ──
# Gabungkan semua label dari kedua dataset untuk LabelEncoder yang konsisten
all_labels = (y_train_imb_str + y_val_imb_str + y_test_imb_str +
              y_train_bal_str + y_val_bal_str + y_test_bal_str)
_, le = encode_labels(all_labels, fit=True)

# Encode imbalance labels
y_train_imb = le.transform(y_train_imb_str)
y_val_imb   = le.transform(y_val_imb_str)

# Encode balance labels
y_train_bal = le.transform(y_train_bal_str)
y_val_bal   = le.transform(y_val_bal_str)

o_index = list(le.classes_).index("O")
print(f"Label classes ({len(le.classes_)}): {list(le.classes_)}")
print(f"O index: {o_index}")

## Cell 5 — Train LightGBM (Imbalance + Balance)

- **Imbalance**: Train pada `imbalance/train.json`, validasi pada `imbalance/val.json`
- **Balance**: Train pada `balance/train.json`, validasi pada `balance/val.json`

In [ ]:
# ── LightGBM Imbalance ──
lgb_imb, lgb_imb_f1, sw_imb = train_lightgbm_imbalance(
    X_train_imb, y_train_imb, y_train_imb_str, X_val_imb, y_val_imb, le
)

# ── LightGBM Balance ──
lgb_bal, lgb_bal_f1, _ = train_lightgbm_balance(
    X_train_bal, y_train_bal, y_train_bal_str, X_val_bal, y_val_bal, le
)

print(f"\n📊 LightGBM Summary:")
print(f"  Imbalance Val F1: {lgb_imb_f1:.4f}")
print(f"  Balance   Val F1: {lgb_bal_f1:.4f}")

## Cell 6 — Train XGBoost (Imbalance + Balance)

- **Imbalance**: Train pada `imbalance/train.json`, validasi pada `imbalance/val.json`
- **Balance**: Train pada `balance/train.json`, validasi pada `balance/val.json`

In [ ]:
# ── Label remapping (XGBoost butuh contiguous 0..N-1) ──
remap_imb, reverse_remap_imb, y_train_imb_remapped = create_label_remap(y_train_imb)

# ── XGBoost Imbalance ──
xgb_imb, xgb_imb_f1 = train_xgboost_imbalance(
    X_train_imb, y_train_imb, y_train_imb_str, X_val_imb, y_val_imb, le,
    remap_imb, reverse_remap_imb, y_train_imb_remapped, sw_imb,
)

# ── XGBoost Balance ──
xgb_bal, xgb_bal_f1, remap_bal, reverse_remap_bal = train_xgboost_balance(
    X_train_bal, y_train_bal, y_train_bal_str, X_val_bal, y_val_bal, le, o_index,
)

print(f"\n📊 XGBoost Summary:")
print(f"  Imbalance Val F1: {xgb_imb_f1:.4f}")
print(f"  Balance   Val F1: {xgb_bal_f1:.4f}")

## Cell 7 — Predict & Evaluate Semua Model

Setiap model diprediksi pada test set dari folder yang sesuai:
- Imbalance models → `imbalance/test.json`
- Balance models → `balance/test.json`

> ⏱️ ~20 menit (threshold tuning + prediction untuk 4 model)

In [ ]:
# Kumpulkan semua model
models_dict = {
    "lgb_imb": lgb_imb,
    "xgb_imb": xgb_imb,
    "lgb_bal": lgb_bal,
    "xgb_bal": xgb_bal,
}

# Jalankan pipeline prediksi lengkap
all_metrics = run_prediction_pipeline(
    models_dict=models_dict,
    # Imbalance test/val data
    X_test_imb=X_test_imb,
    X_val_imb=X_val_imb,
    y_val_imb_str=y_val_imb_str,
    meta_test_imb=meta_test_imb,
    # Balance test/val data
    X_test_bal=X_test_bal,
    X_val_bal=X_val_bal,
    y_val_bal_str=y_val_bal_str,
    meta_test_bal=meta_test_bal,
    # Shared
    le=le,
    o_index=o_index,
    remap_imb=remap_imb,
    reverse_remap_imb=reverse_remap_imb,
    remap_bal=remap_bal,
    reverse_remap_bal=reverse_remap_bal,
)

## Cell 8 — Verifikasi Output Files

In [ ]:
import os

print("📁 Predictions CSV:")
for f in sorted(os.listdir("results/predictions/")):
    if f.endswith(".csv"):
        size = os.path.getsize(f"results/predictions/{f}") / 1024 / 1024
        print(f"  ✅ {f} ({size:.1f} MB)")

print("\n📁 Metrics JSON:")
for f in sorted(os.listdir("results/metrics/")):
    if f.endswith(".json") and f != "metrics_template.json":
        print(f"  ✅ {f}")

# Verifikasi dengan script evaluasi project
print("\n🔍 Verifikasi dengan src/evaluate.py:")
from src.evaluate import evaluate_from_csv
for csv_file in sorted(os.listdir("results/predictions/")):
    if csv_file.endswith(".csv") and ("lightgbm" in csv_file or "xgboost" in csv_file):
        try:
            m = evaluate_from_csv(f"results/predictions/{csv_file}", csv_file.replace("_predictions.csv", ""))
            print(f"  ✅ {csv_file}: Token F1={m['token_level']['f1']:.4f}, Entity F1={m['entity_level']['f1']:.4f}")
        except Exception as e:
            print(f"  ❌ {csv_file}: {e}")

print("\n🎉 Semua output berhasil di-generate!")